# Outliers and Duplicates

## INFO 442 Group 5 Shuyang Zhang 320230942711

In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

diamonds = pd.read_csv('diamonds.csv')

### 1. Load the seaborn diamonds dataset. Use IQR to detect outliers in the price column. How many are flagged?

In [26]:
q1 = diamonds['price'].quantile(0.25)
q3 = diamonds['price'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

price_outliers = ((diamonds['price'] < lower) | (diamonds['price'] > upper)).sum()
print(f"Price outliers (IQR 1.5): {price_outliers}")
print(f"Outlier percentage: {price_outliers/len(diamonds)*100:.1f}%")
print()


Price outliers (IQR 1.5): 3540
Outlier percentage: 6.6%



answer: 3540 rows (6.6% of 53,940 total rows) are flagged as outliers.

### 2. Apply Isolation Forest with contamination=0.02. Do the flagged rows overlap with the IQR flags?

In [29]:
num_cols = ['carat', 'depth', 'table', 'price', 'x', 'y', 'z']
X = diamonds[num_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso_forest = IsolationForest(contamination=0.02, random_state=42)
diamonds['iso_outlier'] = iso_forest.fit_predict(X_scaled) == -1
iso_count = diamonds['iso_outlier'].sum()
print(f"Isolation Forest outliers: {iso_count}")

Isolation Forest outliers: 1079


answer: The overlap between IQR flags and Isolation Forest flags is typically very small

### 3. Winsorize price at the 1st and 99th percentiles. Compare the R² of a linear regression model predicting price before and after winsorizing.

In [38]:
features = ['carat', 'depth', 'table', 'x', 'y', 'z']
X = diamonds[features]

y_orig = diamonds['price']

lower = diamonds['price'].quantile(0.01)
upper = diamonds['price'].quantile(0.99)
y_wins = diamonds['price'].clip(lower=lower, upper=upper)

model = LinearRegression()

scores_orig = cross_val_score(model, X, y_orig, cv=5, scoring='r2')
scores_wins = cross_val_score(model, X, y_wins, cv=5, scoring='r2')

print(f"Original price - R²: {scores_orig.mean():.4f} +/- {scores_orig.std():.4f}")
print(f"Winsorized price - R²: {scores_wins.mean():.4f} +/- {scores_wins.std():.4f}")

Original price - R²: -0.6686 +/- 1.4026
Winsorized price - R²: -0.6865 +/- 1.4333


For diamond price prediction, answer: winsorizing the target variable does not help. Linear regression still performs poorly because the relationship between features and price is non-linear.

### 4. The diamonds dataset has no true duplicates. Artificially inject 50 and verify your deduplication code removes them correctly.

In [13]:
diamonds = pd.read_csv('diamonds.csv')
original_len = len(diamonds)

diamonds_clean = diamonds.drop_duplicates()
clean_len = len(diamonds_clean)
print(f"Clean dataset length: {clean_len}")

np.random.seed(42)
duplicate_indices = np.random.choice(clean_len, 50, replace=False)
duplicates = diamonds_clean.iloc[duplicate_indices].copy()

diamonds_with_dups = pd.concat([diamonds_clean, duplicates], ignore_index=True)
print(f"Shape with duplicates: {diamonds_with_dups.shape}")

diamonds_deduped = diamonds_with_dups.drop_duplicates()
print(f"Shape after deduplication: {diamonds_deduped.shape}")

rows_removed = len(diamonds_with_dups) - len(diamonds_deduped)
print(f"Rows removed: {rows_removed}")


Clean dataset length: 53794
Shape with duplicates: (53844, 10)
Shape after deduplication: (53794, 10)
Rows removed: 50


answer: it worked.